# Broad vs narrow alter–alter ties — u/CaseB

**Hypothesis.** Candidate pairs that meet across *many unrelated subreddits* look strange and draw
platform enforcement. Pairs that meet *often inside one or two subreddits* look like regulars and don't.

Two tie types among the 275 coordination candidates. Pair counts come from their full histories
(`coordination_CaseB_output/alter_alter_diversity_CaseB.parquet`):

- **Broad:** the pair shares posts in `BROAD_MIN_SUBS`+ distinct subreddits.
- **Narrow–frequent:** the pair shares `NARROW_MIN_POSTS`+ posts, all within one subreddit.

The thresholds are set so both graphs have about 150 ties, which keeps the pictures comparable.

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import fisher_exact
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

EGO = 'CaseB'
IN_DIR = 'coordination_CaseB_output'
FIG_DIR = os.path.join('..', 'p2_figures')

BROAD_MIN_SUBS = 3      # broad tie: shared posts span >= 3 subreddits
NARROW_MIN_POSTS = 30   # narrow tie: >= 30 shared posts in exactly 1 subreddit

SURFACE = '#fcfcfb'
MUTED = '#52514e'
INK = '#0b0b0b'
TIE_EDGE = '#7a7873'
EGO_EDGE = '#b5b3ad'
EGO_COLOR = '#0b0b0b'
# account-outcome colours taken from Figures/Reddit_figs/BlackPeopleTwitterclusters1.svg
STATUS_COLORS = {'Active': '#4b2362', 'Suspended': '#eeb182', 'Deleted': '#cfccc4',
                 'not checked': '#ffffff'}
BROAD_COLOR, NARROW_COLOR = '#2a78d6', '#eb6834'

In [ ]:
# Alter-alter ties with their subreddit spread, and the candidates' account status.
# Broad ties span at least BROAD_MIN_SUBS subreddits; narrow ties sit in one subreddit with at least NARROW_MIN_POSTS shared posts.
ties = pd.read_parquet(os.path.join(IN_DIR, 'alter_alter_diversity_CaseB.parquet'))
cands = pd.read_parquet(os.path.join(IN_DIR, 'candidates_CaseB.parquet'))
account_status = cands.set_index('account')['account_status'].to_dict()
base_rate = (cands['account_status'] == 'Suspended').mean()

broad = ties[ties['subreddits'] >= BROAD_MIN_SUBS]
narrow = ties[(ties['subreddits'] == 1) & (ties['posts'] >= NARROW_MIN_POSTS)]


def members(t):
    return set(t['a']) | set(t['b'])


def status_mix(names):
    s = pd.Series([account_status.get(n, 'not checked') for n in names])
    return s.value_counts()


# Accounts touched by each kind of tie and their suspended/deleted shares, against the baseline.
for name, t in (('broad', broad), ('narrow', narrow)):
    m = members(t)
    mix = status_mix(m)
    print(f'{name:7s} ties={len(t):4d} accounts={len(m):3d} '
          f'suspended={mix.get("Suspended", 0)} ({mix.get("Suspended", 0) / len(m):.0%})  '
          f'deleted={mix.get("Deleted", 0)} ({mix.get("Deleted", 0) / len(m):.0%})')
print(f'baseline suspended share across all 275 candidates: {base_rate:.0%}')
both = members(broad) & members(narrow)
print(f'accounts in both graphs: {len(both)}')

### Does the pattern hold across thresholds?
Any single cut-off could be cherry-picked. This sweeps both definitions and shows the suspended share
among the accounts each one touches.

In [ ]:
# Suspended share among accounts in broad ties (1-5+ subreddits) and narrow ties (1-75+ posts), across thresholds.
def sweep():
    rows = []
    for k in range(1, 6):
        t = ties[ties['subreddits'] >= k]
        m = members(t)
        rows.append(('broad', f'≥{k} subreddits', k, len(t), len(m),
                     status_mix(m).get('Suspended', 0) / len(m)))
    for n in (1, 5, 10, 20, 30, 50, 75):
        t = ties[(ties['subreddits'] == 1) & (ties['posts'] >= n)]
        m = members(t)
        rows.append(('narrow', f'≥{n} posts, 1 subreddit', n, len(t), len(m),
                     status_mix(m).get('Suspended', 0) / len(m)))
    return pd.DataFrame(rows, columns=['case', 'threshold', 'level', 'ties', 'accounts', 'suspended_share'])


sweep_df = sweep()
sweep_df

In [ ]:
# Figure: the threshold sweep, one panel per tie kind, with the all-candidate baseline dashed.
def draw_sweep(sweep_df):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    fig.patch.set_facecolor(SURFACE)
    for ax, (case, color, xlabel) in zip(axes, (
            ('broad', BROAD_COLOR, 'minimum distinct subreddits the pair shares posts in'),
            ('narrow', NARROW_COLOR, 'minimum shared posts (pair confined to one subreddit)'))):
        d = sweep_df[sweep_df['case'] == case]
        ax.set_facecolor(SURFACE)
        ax.axhline(base_rate, color=MUTED, lw=1, ls=(0, (4, 3)), zorder=1)
        x = np.arange(len(d))
        ax.plot(x, d['suspended_share'], color=color, lw=2, zorder=2)
        ax.scatter(x, d['suspended_share'], s=64, color=color, edgecolors=SURFACE,
                   linewidths=2, zorder=3)
        for xi, (_, r) in zip(x, d.iterrows()):
            ax.annotate(f'{r["suspended_share"]:.0%}\nn={r["accounts"]}', (xi, r['suspended_share']),
                        xytext=(0, 9), textcoords='offset points', ha='center', va='bottom',
                        fontsize=7.5, color=MUTED)
        ax.set_xticks(x, d['level'])
        ax.set_xlabel(xlabel, fontsize=8.5, color=MUTED)
        ax.tick_params(colors=MUTED, labelsize=8)
        for side in ('top', 'right'):
            ax.spines[side].set_visible(False)
        for side in ('left', 'bottom'):
            ax.spines[side].set_color('#d6d4cf')
        ax.yaxis.grid(True, color='#e8e6e1', lw=0.6); ax.set_axisbelow(True)
        ax.set_title('Broad ties' if case == 'broad' else 'Narrow, frequent ties',
                     loc='left', fontsize=11, color=INK)
    axes[0].set_ylabel('share of accounts suspended', fontsize=8.5, color=MUTED)
    axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    axes[0].set_ylim(0, 0.55)
    axes[0].annotate(f'all 275 candidates: {base_rate:.0%}', (axes[0].get_xlim()[1], base_rate),
                     xytext=(-2, -5), textcoords='offset points', ha='right', va='top',
                     fontsize=7.5, color=MUTED)
    fig.suptitle(f'Suspension rises with tie breadth and falls with tie frequency — u/{EGO} candidates',
                 x=0.01, ha='left', fontsize=12, color=INK)
    fig.tight_layout()
    return fig


fig = draw_sweep(sweep_df)
fig.savefig(os.path.join(FIG_DIR, f'breadth_vs_frequency_sweep_{EGO}.png'), dpi=150, facecolor=SURFACE)
plt.show()

### The two networks
Same drawing for both panels. Fill is the account's current outcome, node size is its degree in that
graph, and edge width is shared posts on a log scale. Every node is an alter of u/CaseB, so ego
spokes are left out: they would hide the tie structure being compared. The largest connected component
fills the left of each panel. Smaller components stack on the right, tagged with the subreddits their
ties run through.

In [ ]:
# Figure helpers and the two-panel network figure (broad vs narrow ties); fill = account status, size = degree.
def tie_graph(t):
    G = nx.Graph()
    for _, r in t.iterrows():
        G.add_edge(r['a'], r['b'], posts=int(r['posts']), subs=r['subreddit_list'].split(','))
    return G


def component_layout(G):
    """Main component in the unit square; the rest stacked in a column to its right."""
    comps = sorted(nx.connected_components(G), key=lambda c: (-len(c), sorted(c)[0]))
    pos, boxes = {}, []
    main = G.subgraph(comps[0])
    p = nx.kamada_kawai_layout(main)
    xy = np.array(list(p.values()))
    lo, span = xy.min(axis=0), np.ptp(xy, axis=0).max()
    pos.update({n: (v - lo) / span for n, v in p.items()})
    rest = comps[1:]
    heights = np.array([np.sqrt(len(c)) for c in rest])
    heights = heights / heights.sum() if len(rest) else heights
    top = 1.0
    for c, h in zip(rest, heights):
        sub = G.subgraph(c)
        p = nx.kamada_kawai_layout(sub) if len(c) > 2 else {n: np.array([i, 0.0]) for i, n in enumerate(sorted(c))}
        xy = np.array(list(p.values()))
        lo, span = xy.min(axis=0), max(np.ptp(xy, axis=0).max(), 1e-9)
        size = min(h * 0.8, 0.22)
        for n, v in p.items():
            pos[n] = np.array([1.2, top - h / 2]) + ((v - lo) / span - 0.5) * size
        boxes.append((c, np.array([1.2, top - h / 2]), size))
        top -= h
    return pos, comps[0], boxes


def top_subs(G, nodes, k=2):
    s = pd.Series([sub for u, v, d in G.subgraph(nodes).edges(data=True) for sub in d['subs']])
    return ', '.join('r/' + x for x in s.value_counts().index[:k])


def declutter(ax, labels, step=4, tries=5):
    """Greedy: push each label further along its own direction until it clears the ones already placed."""
    renderer = ax.figure.canvas.get_renderer()
    placed = []
    for lab in labels:
        dx, dy = lab.xyann
        norm = max(np.hypot(dx, dy), 1e-9)
        for i in range(tries):
            bb = lab.get_window_extent(renderer).expanded(1.0, 1.05)
            if not any(bb.overlaps(p) for p in placed):
                break
            lab.xyann = (dx + (i + 1) * step * dx / norm, dy + (i + 1) * step * dy / norm)
        placed.append(lab.get_window_extent(renderer))


def draw_panel(ax, G, title, max_labels=16, label_fn=None):
    ax.set_facecolor(SURFACE)
    pos, main, boxes = component_layout(G)
    deg = dict(G.degree())
    dmax = max(deg.values())
    posts = [d['posts'] for _, _, d in G.edges(data=True)]
    wmin, wmax = min(posts), max(posts)
    for u, v, d in sorted(G.edges(data=True), key=lambda e: e[2]['posts']):
        f = (np.log(d['posts']) - np.log(wmin)) / max(np.log(wmax) - np.log(wmin), 1e-9)
        ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], color=TIE_EDGE,
                lw=0.6 + 3.2 * f, alpha=0.35 + 0.5 * f, solid_capstyle='round', zorder=2)
    nodes = sorted(G.nodes())
    ax.scatter([pos[n][0] for n in nodes], [pos[n][1] for n in nodes],
               s=[45 + 280 * np.sqrt(deg[n] / dmax) for n in nodes],
               c=[STATUS_COLORS[account_status.get(n, 'not checked')] for n in nodes],
               edgecolors=MUTED, linewidths=0.8, zorder=3)
    # main component: label the best-connected accounts plus every suspended account
    centre = np.array([0.5, 0.5])
    ranked = sorted(main, key=lambda n: (-deg[n], n))
    to_label = ranked[:max_labels] + [n for n in ranked[max_labels:] if account_status.get(n) == 'Suspended']
    for n in to_label:
        x, y = pos[n]
        ang = np.arctan2(y - centre[1], x - centre[0])
        (ax.annotate(label_fn(n, deg[n]) if label_fn else f'{n} ({deg[n]})', (x, y), xytext=(9 * np.cos(ang), 9 * np.sin(ang)),
                                  textcoords='offset points', ha='left' if np.cos(ang) >= 0 else 'right',
                                  va='center', fontsize=6.8, color=INK, zorder=6,
                                  bbox=dict(boxstyle='round,pad=0.1', fc=SURFACE, ec='none', alpha=0.8),
                                  arrowprops=dict(arrowstyle='-', color=EGO_EDGE, lw=0.5, shrinkA=0, shrinkB=4)))
    ax.annotate(f'largest component: {len(main)} accounts\nmostly {top_subs(G, main, 3)}',
                (0.0, 1.04), fontsize=8, color=MUTED, va='bottom', ha='left')
    # small components: one caption each, naming the members and where they meet
    for c, (cx, cy), size in boxes:
        ax.annotate(f'{", ".join(sorted(c))}\n{top_subs(G, c)}', (cx + size / 2 + 0.03, cy),
                    fontsize=6.6, color=MUTED, va='center', ha='left', zorder=6)
    mix = status_mix(nodes)
    handles = [Line2D([], [], marker='o', ls='', mfc=STATUS_COLORS[st], mec=MUTED, mew=0.9,
                      markersize=9, label=f'{st}  {mix[st]} ({mix[st] / len(nodes):.0%})')
               for st in ('Active', 'Suspended', 'Deleted') if mix.get(st)]
    handles.append(Line2D([], [], color=TIE_EDGE, lw=2.5, label=f'tie, width = shared posts ({wmin}–{wmax})'))
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(0.0, -0.01), frameon=False,
              fontsize=8, ncol=2)
    ax.set_title(title, loc='left', fontsize=11.5, color=INK, pad=4)
    ax.set_xlim(-0.25, 1.95); ax.set_ylim(-0.1, 1.1)
    ax.set_aspect('equal'); ax.set_axis_off()


G_broad, G_narrow = tie_graph(broad), tie_graph(narrow)


def draw_networks(fname, label_fn=None, note=''):
    fig, axes = plt.subplots(1, 2, figsize=(24, 8.6))
    fig.patch.set_facecolor(SURFACE)
    draw_panel(axes[0], G_broad,
               f'Broad: pairs sharing posts across ≥{BROAD_MIN_SUBS} subreddits  '
               f'({G_broad.number_of_edges()} ties, {G_broad.number_of_nodes()} accounts)', label_fn=label_fn)
    draw_panel(axes[1], G_narrow,
               f'Narrow, frequent: ≥{NARROW_MIN_POSTS} shared posts, all in one subreddit  '
               f'({G_narrow.number_of_edges()} ties, {G_narrow.number_of_nodes()} accounts)', label_fn=label_fn)
    fig.suptitle(f'Two kinds of alter–alter tie among u/{EGO}\'s 275 coordination candidates  '
                 f'(all candidates: {base_rate:.0%} suspended){note}', x=0.01, ha='left', fontsize=13, color=INK)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.canvas.draw()                      # layout is final: now nudge the labels apart
    for ax in axes:
        declutter(ax, [t for t in ax.texts if t.arrow_patch is not None])
    # equal aspect leaves slack above the panels: pull the headline down onto them
    fig._suptitle.set_y(axes[0].get_position().y1 + 0.1)
    fig.savefig(os.path.join(FIG_DIR, fname), dpi=150, facecolor=SURFACE, bbox_inches='tight', pad_inches=0.3)
    plt.show()


# label = (ties in this graph)
draw_networks(f'breadth_vs_frequency_networks_{EGO}.png')

### Same networks, labelled with closeness to u/CaseB
Label = `account (ties in this graph / distinct posts shared with u/CaseB)`. The second number is
the `contacts` column from the candidate table.

In [ ]:
# Same figure, labels add posts shared with the ego; and how close each graph's accounts are to the ego.
ego_contacts = cands.set_index('account')['contacts'].to_dict()
draw_networks(f'breadth_vs_frequency_networks_ego_contacts_{EGO}.png',
              label_fn=lambda n, d: f'{n} ({d} / {ego_contacts.get(n, 0)})',
              note=f'\nlabels: account (ties in this graph / posts shared with u/{EGO})')

for name, G in (('broad', G_broad), ('narrow', G_narrow)):
    c = pd.Series([ego_contacts[n] for n in G.nodes()])
    print(f'{name:7s} posts shared with u/{EGO}: median {c.median():.0f}, mean {c.mean():.1f}, max {c.max()}')

### What the broad ties span, and where the narrow ones sit

In [ ]:
# The widest broad ties, and which subreddits the narrow ties sit in.
print('broad ties, widest first:')
display(broad.sort_values(['subreddits', 'posts'], ascending=False)
        .assign(status_a=lambda d: d['a'].map(account_status), status_b=lambda d: d['b'].map(account_status))
        [['a', 'status_a', 'b', 'status_b', 'posts', 'subreddits', 'subreddit_list']].head(25))
print('narrow ties by subreddit:')
display(narrow['subreddit_list'].value_counts().head(15))

### Account-level test
Tie-level graphs can double-count hub accounts. This compares accounts directly. An account counts as
*broad* if it has any broad tie, and as *narrow* if it has any narrow tie but no broad one.

In [ ]:
# Account-level test: suspended share for accounts with any broad tie versus narrow-only accounts (Fisher's exact test).
acct = pd.DataFrame({'account': cands['account'], 'status': cands['account_status']})
acct['group'] = np.where(acct['account'].isin(members(broad)), 'broad',
                 np.where(acct['account'].isin(members(narrow)), 'narrow only', 'neither'))
tab = pd.crosstab(acct['group'], acct['status'], margins=True)
display(tab)
b = acct[acct['group'] == 'broad']['status'].eq('Suspended')
n = acct[acct['group'] == 'narrow only']['status'].eq('Suspended')
odds, p = fisher_exact([[b.sum(), (~b).sum()], [n.sum(), (~n).sum()]])
print(f'suspended: broad {b.mean():.0%} (n={len(b)}) vs narrow-only {n.mean():.0%} (n={len(n)}); '
      f'odds ratio {odds:.2f}, Fisher p={p:.3f}')